# Trace Count v26/v26.1: 256-token diverse-set untied-head control

This paired experiment returns to the 256-character v24.3 regime, where the
historical held-out result was Thinking 0.632 versus Non-thinking 0.372.  Both
modes use the same 100 three-character marker sets, exactly count-balanced
training sampler, 4-layer/4-head transformer, separator/no-index trace,
component-normalized loss, optimizer, seed, examples, and 10,000-step
schedule.

The sole training change from v24.3 is an untied native LM head.  Its weights
are copied from the input embedding at step zero, so the initial function is
identical; later atomic-number output gradients cannot distort the input
number embeddings.  No answer-query contrastive or probe loss is used.  NCC
therefore remains a post-hoc measurement of an emergent count representation.

V26.1 then applies one validation-selected number-row calibration schedule to
both frozen backbones.  It can align an existing representation with native
number tokens, but cannot create missing retrieval or count information.  The
primary gate requires high, count-uniform Thinking accuracy, trace fidelity,
and a positive held-out Thinking-minus-Non-thinking accuracy gap.


## 1. Mount Google Drive

In [ ]:
from pathlib import Path

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
DRIVE_READY = False
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive", timeout_ms=300000)
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_READY = True
    print("Drive ready:", DRIVE_RESULTS_ROOT)
else:
    print("Local runtime: Drive mount skipped")


## 2. Clone the audited implementation and verify GPU

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

assert DRIVE_READY, "Run the Drive cell first"
REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
REPO_REF = "agent/remove-misplaced-realistic-artifacts"
preferred = Path("/content/Synthetic_CoT_NiaH_Count")
candidates = [Path.cwd(), *Path.cwd().parents, preferred]
repo = next((path.resolve() for path in candidates if (path / "pyproject.toml").exists()), None)
if repo is None:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(preferred)],
        check=True,
    )
    repo = preferred
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", REPO_REF], check=True)
os.chdir(repo)

probe = subprocess.run(
    [sys.executable, "-c", "import numpy,pandas,scipy,matplotlib,seaborn"],
    capture_output=True,
    text=True,
)
if probe.returncode:
    print(probe.stderr[-2000:])
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "--force-reinstall", "numpy==1.26.4", "pandas==2.2.3",
            "scipy==1.13.1", "matplotlib==3.8.4", "seaborn==0.13.2",
        ],
        check=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Scientific ABI repaired. Reconnect and rerun all cells.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
src_root = str(repo / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
os.environ["PYTHONPATH"] = src_root + os.pathsep + os.environ.get("PYTHONPATH", "")

import codecs
import pandas as pd
import torch
import synthetic_counting_v26
import synthetic_counting_v26_1
from IPython.display import display

def run_streaming(command):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        print(decoder.decode(chunk), end="", flush=True)
    print(decoder.decode(b"", final=True), end="", flush=True)
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print({
    "repo": str(repo),
    "repo_ref": REPO_REF,
    "repo_commit": commit,
    "v26_package": synthetic_counting_v26.__file__,
    "v26_1_package": synthetic_counting_v26_1.__file__,
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
assert torch.cuda.is_available()


## 3. Audit the 256-token paired setting


In [ ]:
from dataclasses import asdict
from synthetic_counting_v26.config import preset_config
from synthetic_counting_v24_3.config import preset_config as v24_3_preset_config

VERSION = "v26"
PRESET = "main"
SEED = 1234
DEVICE = "cuda"
RUN_NAME = "v26_L256_pool100_uniform_count_untied_seed1234"
CALIBRATION_NAME = "v26.1_native_head_calibration_L256_pool100_seed2478"
OUT_ROOT = "runs/synthetic_counting_v26"
CHECKPOINT_SYNC_ROOT = DRIVE_RESULTS_ROOT
SKIP_COMPLETED = True

PLANNED_CONFIG = preset_config(PRESET, seed=SEED, device=DEVICE)
BASELINE_CONFIG = v24_3_preset_config(PRESET, seed=SEED, device=DEVICE)
changed_fields = {
    key for key, value in asdict(PLANNED_CONFIG).items()
    if asdict(BASELINE_CONFIG).get(key) != value
}
assert changed_fields == {"version", "tie_word_embeddings"}, changed_fields
assert PLANNED_CONFIG.seq_len == 256
assert PLANNED_CONFIG.max_render_len == 287
assert PLANNED_CONFIG.n_positions == 384
assert PLANNED_CONFIG.count_max_threshold == 10
assert PLANNED_CONFIG.needle_pool_size == 100
assert PLANNED_CONFIG.needle_pool_frequency_threshold == 10.0 / 256.0
assert PLANNED_CONFIG.training_count_distribution == "uniform"
assert PLANNED_CONFIG.batch_size == 128
assert PLANNED_CONFIG.enabled_model_variants == ("rope/nonthinking", "rope/thinking")
assert PLANNED_CONFIG.trace_format == "separator"
assert PLANNED_CONFIG.task_output_loss_reduction == "component_normalized"
assert not PLANNED_CONFIG.tie_word_embeddings
assert PLANNED_CONFIG.answer_query_contrastive_weight == 0.0
print({
    "controlled_changes_from_v24.3": sorted(changed_fields),
    "sequence_layout": "<BOS> query[5] data[256] output",
    "longest_thinking_sequence": PLANNED_CONFIG.max_render_len,
    "position_budget": PLANNED_CONFIG.n_positions,
    "answer_support": "1..10 atomic tokens",
    "trace": "(<Sep> marker) repeated n times",
    "sampler": "uniform semantic count; natural feasible-set exposure",
    "comparison_scope": "paired within-v26; identical examples and optimization settings",
})

RUN_DIR = Path(OUT_ROOT) / RUN_NAME
DRIVE_RUN_DIR = CHECKPOINT_SYNC_ROOT / RUN_NAME
CALIBRATION_DIR = DRIVE_RESULTS_ROOT / CALIBRATION_NAME


## 4. Prepare and audit the fixed 256-token data


In [ ]:
base_cmd = [
    sys.executable, "-u", "-m", "synthetic_counting_v26.run_v26",
    "--preset", PRESET,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--train-steps", "10000",
    "--max-steps-for-language-pred", "1500",
    "--checkpoint-every", "100",
    "--recovery-every", "500",
    "--snapshot-shard-every", "500",
    "--eval-every", "500",
    "--ar-eval-every", "1000",
    "--ar-examples-per-count", "2",
    "--permutation-examples-per-count", "1",
    "--eval-examples-per-count", "10",
    "--final-examples-per-count", "50",
    "--phase-head-selection-examples-per-count", "2",
    "--phase-examples-per-count", "1",
    "--out-root", OUT_ROOT,
    "--run-name", RUN_NAME,
    "--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT),
]
if SKIP_COMPLETED:
    base_cmd.append("--skip-completed")
run_streaming([*base_cmd, "--stage", "prepare"])

print("Prepared local run:", RUN_DIR.resolve())
print("Drive target:", DRIVE_RUN_DIR)


## 5. Train the paired 256-token models


In [ ]:
training_started = time.perf_counter()
run_streaming([*base_cmd, "--stage", "train"])
print(f"Paired training block: {time.perf_counter() - training_started:.1f} seconds")

sampling = pd.read_csv(RUN_DIR / "tables" / "training_sampling_distribution.csv")
accepted = sampling[sampling["dimension"].eq("accepted_counts")].copy()
count_table = accepted.pivot(index="mode", columns="value", values="examples").sort_index(axis=1)
assert list(count_table.columns.astype(int)) == list(range(1, 11))
assert count_table.loc["nonthinking"].equals(count_table.loc["thinking"])
count_relative_error = (
    count_table.sub(count_table.mean(axis=1), axis=0)
    .abs()
    .div(count_table.mean(axis=1), axis=0)
)
assert float(count_relative_error.to_numpy().max()) < 0.01

set_rows = sampling[sampling["dimension"].eq("set_ids")].copy()
set_table = set_rows.pivot(index="mode", columns="value", values="examples").sort_index(axis=1)
assert set_table.shape == (2, 100)
assert set_table.loc["nonthinking"].equals(set_table.loc["thinking"])
set_cv = set_table.std(axis=1) / set_table.mean(axis=1)
assert float(set_cv.max()) < 0.25
assert int(set_table.min(axis=1).min()) > 0
print({
    "maximum_count_relative_error": float(count_relative_error.to_numpy().max()),
    "set_exposure_cv": set_cv.to_dict(),
    "minimum_set_examples": int(set_table.min(axis=1).min()),
    "maximum_set_examples": int(set_table.max(axis=1).max()),
    "paired_sampling_exactly_matched": True,
})
display(pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_summary.csv"))
display(pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_by_count.csv"))


## 6. Symmetric native-head calibration

The schedule is selected using validation prompts in Thinking mode and then
applied unchanged to the paired Non-thinking checkpoint.  Only the ten native
atomic-number rows can move; both transformers, input embeddings, attention
heads, and all hidden-state geometry remain frozen.  Test prompts are opened
once after both choices are fixed.


In [ ]:
run_streaming([
    sys.executable, "-u", "-m", "synthetic_counting_v26_1.run_v26_1",
    "--source-run", str(DRIVE_RUN_DIR),
    "--output-dir", str(CALIBRATION_DIR),
    "--device", DEVICE,
    "--batch-size", "128",
    "--eval-every", "100",
    "--validation-per-count", "10",
    "--seed", "2478",
])


## 7. Measure count compression in the frozen backbones

In [ ]:
NCC_OUTPUT = RUN_DIR / "analysis" / "aligned_ncc"
run_streaming([
    sys.executable, "-u", "scripts/compare_v24_modes_ncc.py",
    "--results-root", str(RUN_DIR.parent),
    "--output", str(NCC_OUTPUT),
    "--run-prefix", RUN_DIR.name,
    "--expected-version", VERSION,
    "--device", DEVICE,
    "--discovery-per-label", "10",
    "--confirmation-per-label", "8",
    "--batch-size", "32",
])
display(pd.read_csv(NCC_OUTPUT / "selected_confirmation_summary.csv")[[
    "comparison_mode", "endpoint", "layer",
    "chance_balanced_accuracy",
    "confirmation_logistic_balanced_accuracy",
    "confirmation_ncc_balanced_accuracy",
    "confirmation_ncc_above_chance",
]])


## 8. Evaluate the behavioral advantage gate

In [ ]:
summary = pd.read_csv(CALIBRATION_DIR / "final_summary.csv")
by_count = {
    mode: pd.read_csv(CALIBRATION_DIR / "final" / mode / "final_autoregressive_by_count.csv")
    for mode in ("thinking", "nonthinking")
}
display(summary)
for mode, frame in by_count.items():
    print(mode)
    display(frame)

thinking = summary[summary["mode"].eq("thinking")].iloc[0]
nonthinking = summary[summary["mode"].eq("nonthinking")].iloc[0]
accuracy_gap = float(thinking.test_overall_accuracy - nonthinking.test_overall_accuracy)
advantage_gate = bool(
    float(thinking.test_overall_accuracy) >= 0.90
    and float(thinking.test_minimum_count_accuracy) >= 0.85
    and float(thinking.test_trace_exact_accuracy) >= 0.90
    and accuracy_gap >= 0.05
)
result = {
    "thinking_accuracy": float(thinking.test_overall_accuracy),
    "nonthinking_accuracy": float(nonthinking.test_overall_accuracy),
    "thinking_minus_nonthinking": accuracy_gap,
    "thinking_minimum_count_accuracy": float(thinking.test_minimum_count_accuracy),
    "thinking_count_spread": float(thinking.test_count_accuracy_spread),
    "thinking_trace_exact": float(thinking.test_trace_exact_accuracy),
    "behavioral_advantage_gate": advantage_gate,
}
print(result)


## 9. Verify persistence; preserve the runtime if another setting is needed

In [ ]:
import json

required = [
    DRIVE_RUN_DIR / "config.json",
    DRIVE_RUN_DIR / "manifest.json",
    DRIVE_RUN_DIR / "checkpoints" / "rope" / "nonthinking" / "final" / "checkpoint.pt",
    DRIVE_RUN_DIR / "checkpoints" / "rope" / "thinking" / "final" / "checkpoint.pt",
    CALIBRATION_DIR / "manifest.json",
    CALIBRATION_DIR / "final_summary.csv",
    CALIBRATION_DIR / "final" / "nonthinking" / "checkpoint.pt",
    CALIBRATION_DIR / "final" / "thinking" / "checkpoint.pt",
]
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]
assert not missing, missing
manifest = json.loads((CALIBRATION_DIR / "manifest.json").read_text())
assert manifest["status"] == "complete"
assert manifest["experiment"] == "v26.1"
assert manifest["source_version"] == "v26"
print("Drive persistence verified:", DRIVE_RUN_DIR, CALIBRATION_DIR)

if advantage_gate and Path("/content").exists():
    print("Behavioral advantage gate passed; disconnecting in 10 seconds.")
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Advantage gate did not pass; keeping the runtime available for the next controlled setting.")
